In [34]:
import json
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from collections import defaultdict
import json


In [35]:
A = ["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"],
    "TT": ["Tetanus"],
    "HepB": ["Hepatitis_B"],
    "Hib": ["Hib"],
    "IPV": ["Polio"],
    "OPV": ["Polio"],
    "DT": ["Diphtheria", "Tetanus"],
    "Td": ["Diphtheria", "Tetanus"],
    "DTwP": ["Diphtheria", "Tetanus", "Pertussis"],
    "DTwP-Hib": ["Diphtheria", "Tetanus", "Pertussis", "Hib"],
    "Penta": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib"],
    "Hexa": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"],
    "HPV": ["HPV"],
    "Rotavirus": ["Rotavirus"],
    "PCV": ["PCV"]
}

P = [
    "AJ_Vaccines",
    "BB_NCIPD",
    "China_National",
    "Bharat_Biotech",
    "Bilthoven",
    "Biological_E",
    "GSK",
    "Haffkine_Bio",
    "LG_Chem",
    "Merck_Sharp",
    "Panacea_Biotec",
    "PT_Bio",
    "Sanofi",
    "Serum_Institute",
    "Pfizer"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"],
    "TT": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "HepB": ["Serum_Institute", "LG_Chem"],
    "Hib": ["Serum_Institute"],
    "IPV": ["LG_Chem", "AJ_Vaccines", "Bilthoven", "Sanofi"],
    "OPV": ["Serum_Institute", "PT_Bio", "GSK", "Sanofi", "Panacea_Biotec", "China_National", "Bharat_Biotech", "Haffkine_Bio"],
    "DT": ["PT_Bio", "BB_NCIPD"],
    "Td": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "DTwP": ["Serum_Institute", "Biological_E"],
    "DTwP-Hib": ["Serum_Institute"],
    "Penta": ["Serum_Institute", "PT_Bio", "Biological_E", "LG_Chem", "Panacea_Biotec"],
    "Hexa": ["Sanofi"],
    "HPV": ["GSK", "Merck_Sharp", "China_National"],
    "Rotavirus": ["Serum_Institute", "GSK", "Bharat_Biotech"],
    "PCV": ["Serum_Institute", "GSK", "Pfizer"]
}



## Define values

In [36]:
# define constants
beta = 10.0  

tmin = 1
tmax = 10
# Define parameters
tmin = 1
tmax = 10 
beta = 10
max_tender_length = 5

Δ = [i for i in range(1, max_tender_length + 1)]

# Generate time periods
T = list(range(tmin, tmax + 1))

# Calculate delta values
delta = {t: (1 + 0.03) ** t for t in T}

#  Tender cost
g = {t: 1e8 for t in T}

# Cost of expanding capacity for each producer
gamma = {p: 1e8 for p in P}  

#  beta
beta = {t: beta for t in T}

# Inventory holding cost
h = {v: 0.01 for v in V}  

F_time_set = []

for t in T:
    for tau in T:
        if tau >= t:
            if (tau - t + 1) in Δ:
                F_time_set.append((t, tau))






## Read in necessary data

In [37]:
# import start data
filename = "data/Starting_point.xlsx"
starting_points_file = pd.read_excel(filename, sheet_name="F_start")

starting_points_vect_F = [
    (row['Antigen'], (row['Starting'], row['Ending']))
    for _, row in starting_points_file.iloc[1:].iterrows()
]

#scenario probabilities
with open('data/scenario_pair_probabilities_new.json', 'r') as f:
    probabilities = json.load(f)

# import results
# Define the path to the file
# file_path = 'social_surplus_base.json'
file_path = 'social_surplus_base.json'

# Load the JSON file
with open(file_path, 'r') as file:
    data = json.load(file)

### read in and transform price data to dict

In [38]:

# Load the Excel file to examine its structure
file_path = 'data/Vaccine_price_data.xlsx'
xlsx = pd.ExcelFile(file_path)

# Get all sheet names and skip the first two sheets
sheet_names = xlsx.sheet_names[2:]

# Create a nested dictionary with structure: vaccine[producer][year]
vaccine_dict = {}

for sheet in sheet_names:
    # Read each sheet
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Create a nested dictionary for each sheet
    sheet_dict = {}
    for _, row in df.iterrows():
        producer = row['Unnamed: 0'] if 'Unnamed: 0' in row else None
        if producer:
            # Initialize dictionary for each producer
            if producer not in sheet_dict:
                sheet_dict[producer] = {}

            # Populate year data
            for col in df.columns:
                if isinstance(col, int):  # Assuming year columns are integers
                    sheet_dict[producer][col] = row[col]

    # Add sheet's nested dictionary to vaccine dictionary
    vaccine_dict[sheet] = sheet_dict

# Displaying a small portion of the resulting nested dictionary structure
vaccine_dict_sample = {sheet: list(vaccine_dict[sheet].items()) for sheet in vaccine_dict}
modified_dict = {}

for key, value in vaccine_dict_sample.items():
    # Split the key by space and keep only the first part
    new_key = key.split()[0]
    # Add the new key with the original value to the new dictionary
    modified_dict[new_key] = value

# Replace the original dictionary with the modified one
vaccine_price_dict = modified_dict

for vaccine, producers_list in vaccine_price_dict.items():
    # Convert list of tuples to a dictionary
    producers_dict = dict(producers_list)
    # Replace the list with the newly created dictionary
    vaccine_price_dict[vaccine] = producers_dict


## Calculate Tender costs

In [39]:
f_data = data['F']
# Dictionary to store results in the format F[a][(t, tau)] = 1
f_data_modified = {}

# Function to save results in the specified format F[a][(t, tau)] = 1
def find_ones_in_f_modified(data, path=""):
    if isinstance(data, dict):
        for key, value in data.items():
            find_ones_in_f_modified(value, f"{path}.{key}" if path else key)
    elif isinstance(data, list):
        for index, value in enumerate(data):
            find_ones_in_f_modified(value, f"{path}[{index}]")
    elif data == 1:
        keys = path.split('.')
        a = keys[0]
        t_tau = tuple(map(int, keys[1:]))
        
        if a not in f_data_modified:
            f_data_modified[a] = {}
        
        f_data_modified[a][t_tau] = 1

# Execute the function for "F" data
find_ones_in_f_modified(f_data)

# Remove initial conditions from f_data_modified
for antigen, t_tau in starting_points_vect_F:
    if antigen in f_data_modified and t_tau in f_data_modified[antigen]:
        del f_data_modified[antigen][t_tau]

# Update the summation after removing initial conditions
f_data_summed = {}

# Calculate the sum for each antigen based on the given formula
for antigen, values in f_data_modified.items():
    total_sum = 0
    for (t, tau), f_value in values.items():
        # Applying the formula: g[t] * F[a][(t, tau)] / delta[t]
        total_sum += (g[t] * f_value) / delta[t]
        
    # Store the summed value for each antigen
    f_data_summed[antigen] = total_sum

# Calculate the overall summation across all antigens
F_OBJ_Value = sum(f_data_summed.values())

# Display the total sum for all antigens
print(f"F Objective cost: {F_OBJ_Value}")


F Objective cost: 2081481746.8186831


## Calculate Capacity Extension Costs

In [40]:
# Assuming the "L" key contains the data you mentioned
L_data = data.get("L", {})

# Reverse the dictionary structure from producer[year] to year[producer]
L_reversed_data = {}
for producer, years in L_data.items():
    for year, value in years.items():
        if year not in L_reversed_data:
            L_reversed_data[year] = {}
        # L_reversed_data[year][producer] = value
        L_reversed_data[year][producer] = (L_reversed_data[year][producer] * gamma[producer]) / delta[int(year)]

yearly_sums = {year: sum(producer_values.values()) for year, producer_values in L_reversed_data.items()}

# Calculating the sum of all years
L_OBJ_Value = sum(yearly_sums.values())

## Calculate missed doses by scenario - OBJ Function calcs are different for each model

In [41]:
S_data = data['S']
S_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

# Iterate through current structure and move the scenario to the top level
for antigen, years in S_data.items():
    for year, scenarios in years.items():
        for scenario, value in scenarios.items():
            S_data_by_scenario[scenario][year][antigen] = value

# Convert to a regular dictionary for easy use
S_data_by_scenario = dict(S_data_by_scenario)

In [42]:
for scenario, years in S_data_by_scenario.items():
    for year, antigens in years.items():
        if int(year) in beta:  # Ensure the year is in the beta dictionary
            beta_factor = beta[int(year)]
            for antigen in antigens.keys():
                if file_path == 'min_unvax_base.json':
                    #for min unvax model
                    S_data_by_scenario[scenario][year][antigen] = (S_data_by_scenario[scenario][year][antigen] * beta[int(year)]) / delta[int(year)]
                else:
                    # max social surplus value
                    S_data_by_scenario[scenario][year][antigen] = (S_data_by_scenario[scenario][year][antigen] * beta[int(year)]) / delta[int(year)]
                

In [43]:
# Calculating the sum of values under 'S' by scenario
scenario_sums = {}

# Iterate through each scenario to calculate the sum of values under 'S'
for scenario, years in S_data_by_scenario.items():
    scenario_sum = 0
    for year in years.values():
        for antigen, value in year.items():
            if isinstance(value, (int, float)):  # Ensure the value is numeric
                scenario_sum += value
    scenario_sums[scenario] = scenario_sum


In [44]:
S_OBJ_Value = {k: probabilities[k] * scenario_sums[k] for k in probabilities}
print(f"Missed Dose OBJ Value: {sum((S_OBJ_Value.values()))}")

Missed Dose OBJ Value: 13996956.35347403


## Calculate doses purchased

In [45]:
X_data = data['X']
X_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

# Iterate through the original structure to rearrange the keys with the correct nesting
for vaccine, producers in X_data.items():
    for producer, years in producers.items():
        for year, scenarios in years.items():
            for scenario, value in scenarios.items():
                # Move scenario to the top level, followed by year, vaccine, and then producer
                X_data_by_scenario[scenario][year][vaccine][producer] = value

# Convert to a regular dictionary for easy use
X_data_by_scenario = dict(X_data_by_scenario)

In [46]:
X_data_by_scenario

for scenario, year_data in X_data_by_scenario.items():
    for years, vaccine_data in year_data.items():
        for vaccines, producer_data in vaccine_data.items():
            for producer, purchases in producer_data.items():
                # print(f"vaccine: {vaccines}, producer: {producer}, year: {years}, scenario: {scenario} purchase: {purchases} ")
                # print(f"Initial Value: {X_data_by_scenario[scenario][year][vaccines][producer]}")
                # print(f"Vaccine cost: {vaccine_price_dict[vaccines][producer][int(year)]}")
                X_data_by_scenario[scenario][year][vaccines][producer] = X_data_by_scenario[scenario][year][vaccines][producer] * vaccine_price_dict[vaccines][producer][int(year)] / delta[int(year)]


In [56]:
def calculate_scenario_sums(data):
    vax_scenario_sums = {}
    for scenario, years in data.items():
        total_sum = 0
        for year, producers in years.items():
            for vaccine, producers_data in producers.items():
                total_sum += sum(producers_data.values())
        vax_scenario_sums[scenario] = total_sum
    return scenario_sums

# Calculate and print the scenario sums
vax_scenario_sums = calculate_scenario_sums(X_data_by_scenario)


In [57]:
X_OBJ_Value = {k: probabilities[k] * vax_scenario_sums[k] for k in probabilities}
print(f"Vaccines Purchased OBJ Value: {sum((X_OBJ_Value.values()))}")

Vaccines Purchased OBJ Value: 483418379516.37177


## calculate inventory holding costs

In [61]:
I_data = data.get("I", {})

reversed_I_data = {}

for vaccine, years_data in I_data.items():
    for year, scenarios_data in years_data.items():
        for scenario, value in scenarios_data.items():
            if scenario not in reversed_I_data:
                reversed_I_data[scenario] = {}
            if year not in reversed_I_data[scenario]:
                reversed_I_data[scenario][year] = {}
            reversed_I_data[scenario][year][vaccine] = value

reversed_I_data

KeyError: '5'

# TOTAL OBJ VALUE

In [ ]:
F_OBJ_Value + L_OBJ_Value + X_OBJ_Value + S_OBJ_Value + I_OBJ_Value